# Notebook 00: Single-Tool SRE Agent with Amazon Bedrock

## Overview

In this notebook, you’ll create a simple AI-powered Site Reliability Engineering (SRE) agent using Amazon Bedrock and Strands Agents. The agent will leverage a single tool to query a FastAPI backend that simulates a Kubernetes pod-status API, then use a foundation model to diagnose pod failures.

This notebook establishes the foundational concepts behind tool-enabled agents and introduces the core Strands tool decorator pattern.


### Learning Objectives

By the end of this notebook, you will be able to:
- Stand up a local FastAPI server simulating Kubernetes pod data  
- Define a custom tool (`get_pod_status`) the Strands Agents library to fetch pod health  
- Configure a basic agent that uses a foundation model via Amazon Bedrock



### Prerequisites

**AWS Requirements:**
- AWS Account with Amazon Bedrock access
- AWS CLI credentials configured in your environment
- Cross-region inference profile of Claude 3.7 Sonnet model enabled 

**Technical Requirements:**
- Python 3.12+  
- Internet connectivity for package installation


### Architecture Overview
##### Architecture: Single-Tool Agent Flow

```text
User prompt (“Why is the payment API crashing?”)
        │
        ▼
Strands Agents (Claude 3.7 Sonnet via Bedrock)
        │ invokes
        ▼
@get_pod_status() tool ──► FastAPI backend (localhost:8011/pods)
        │ returns JSON list of pods
        ▼
Agent processes tool output and returns diagnostic response
```

### What You’ll Build

A self-contained troubleshooting workflow where:

1. **User** asks “Why is pod X crashing?”  
2. **Agent** calls `get_pod_status`  
3. **Tool** fetches pod data from FastAPI  
4. **Agent** analyzes results and explains which pods are unhealthy


## Step 1: Environment Validation and Setup

First, let's validate your environment and install required packages.

In [ ]:
%%bash
pip install --quiet \
    fastapi \
    uvicorn[standard] \
    strands-agents \
    requests \
    boto3 \
    pydantic

echo "✅ Required packages installed successfully"

In [ ]:
import boto3
from botocore.exceptions import NoCredentialsError, ClientError

REGION = "us-east-1"  # Change if you are using a different region

def check_aws_environment():
    """Validate AWS credentials"""
    try:
        sts = boto3.client("sts", region_name=REGION)
        identity = sts.get_caller_identity()
        account_id = identity.get("Account", "Unknown")
        print(f"✅ AWS credentials OK — Account: {account_id}, Region: {REGION}")
        return account_id, REGION
    except NoCredentialsError:
        raise RuntimeError("❌ AWS credentials not configured — run `aws configure`")
    except ClientError as e:
        raise RuntimeError(f"❌ AWS API error: {e.response['Error']['Code']}")
    except Exception as e:
        raise RuntimeError(f"❌ Unexpected error: {e}")

def check_enabled_models(region=REGION):
    """Check if specific models are enabled"""
    bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

    # Models to test
    test_models = [
        "us.anthropic.claude-3-7-sonnet-20250219-v1:0",  # Update list as needed
    ]

    enabled_models = []
    for model_id in test_models:
        try:
            bedrock_runtime.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": "Hello"}]}],
                inferenceConfig={"maxTokens": 5}
            )
            enabled_models.append(model_id)
            print(f"✅ {model_id} is ENABLED")
        except ClientError as e:
            error_code = e.response["Error"]["Code"]
            if error_code == "AccessDeniedException":
                print(f"❌ {model_id} is NOT ENABLED")
            else:
                print(f"⚠️  {model_id} - {error_code}")
    return enabled_models

# --- Usage ---
print("- AWS Environment Validation")
account, region = check_aws_environment()

print("\n- Checking Bedrock model availability via Converse API")
enabled_models = check_enabled_models(region)
print(f"\nEnabled models: {enabled_models}")


- AWS Environment Validation
✅ AWS credentials OK — Account: 533267284022, Region: us-east-1

- Checking Bedrock model availability via Converse API
✅ us.anthropic.claude-3-7-sonnet-20250219-v1:0 is ENABLED

Enabled models: ['us.anthropic.claude-3-7-sonnet-20250219-v1:0']


In [ ]:
# Import required libraries with error handling
import os
import time
import threading
import requests
from typing import Dict, Any, Optional

try:
    from fastapi import FastAPI, HTTPException
    from strands import Agent, tool
    from strands.models import BedrockModel
    import uvicorn
    print("✅ All libraries imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure all packages are installed correctly")
    raise

## Step 2: Create Infrastructure Simulation Backend

We'll create a FastAPI backend that simulates a Kubernetes cluster with realistic pod data representing common failure scenarios.

In [ ]:
# Create FastAPI application with comprehensive Kubernetes simulation
app = FastAPI(
    title="Kubernetes API Simulator",
    description="Simulates Kubernetes API for SRE agent training",
    version="1.0.0"
)

# Realistic pod data representing various failure scenarios
PODS_DATA = {
    "pods": [
        {
            "name": "payment-service-7d4f8-x5m1q",
            "namespace": "production",
            "status": "CrashLoopBackOff",
            "ready": False,
            "restart_count": 15,
            "cpu_usage": "25%",
            "memory_usage": "98%",
            "node": "worker-node-2",
            "last_restart": "2024-01-15T14:24:30Z",
            "age": "2h15m",
            "containers": [
                {
                    "name": "payment-api",
                    "image": "payment-service:v1.2.3",
                    "status": "Waiting",
                    "reason": "CrashLoopBackOff",
                    "message": "Back-off 5m0s restarting failed container",
                    "exit_code": 137
                }
            ],
            "events": [
                "Warning: OutOfMemoryError in container payment-api",
                "Warning: Back-off restarting failed container",
                "Error: Container payment-api failed with exit code 137",
                "Warning: Failed to pull image payment-service:v1.2.3"
            ],
            "resource_limits": {
                "cpu": "500m",
                "memory": "512Mi"
            },
            "resource_requests": {
                "cpu": "250m",
                "memory": "256Mi"
            }
        },
        {
            "name": "user-service-9k2x1-y6n2r",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 0,
            "cpu_usage": "32%",
            "memory_usage": "64%",
            "node": "worker-node-1",
            "last_restart": None,
            "age": "5d2h",
            "containers": [
                {
                    "name": "user-api",
                    "image": "user-service:v1.1.0",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container started successfully",
                    "exit_code": None
                }
            ],
            "events": [
                "Normal: Successfully pulled image user-service:v1.1.0",
                "Normal: Created container user-api",
                "Normal: Started container user-api"
            ],
            "resource_limits": {
                "cpu": "1000m",
                "memory": "1Gi"
            },
            "resource_requests": {
                "cpu": "500m",
                "memory": "512Mi"
            }
        },
        {
            "name": "database-service-abc123-def456",
            "namespace": "production",
            "status": "Pending",
            "ready": False,
            "restart_count": 0,
            "cpu_usage": "0%",
            "memory_usage": "0%",
            "node": None,
            "last_restart": None,
            "age": "10m",
            "containers": [
                {
                    "name": "postgres",
                    "image": "postgres:14",
                    "status": "Waiting",
                    "reason": "PodScheduled",
                    "message": "0/3 nodes are available: insufficient memory",
                    "exit_code": None
                }
            ],
            "events": [
                "Warning: FailedScheduling - 0/3 nodes are available: insufficient memory",
                "Normal: Scheduled - Successfully assigned to worker-node-3"
            ],
            "resource_limits": {
                "cpu": "2000m",
                "memory": "4Gi"
            },
            "resource_requests": {
                "cpu": "1000m",
                "memory": "2Gi"
            }
        }
    ]
}

@app.get("/health")
def health_check() -> Dict[str, str]:
    """Health check endpoint for the Kubernetes API simulator"""
    return {
        "status": "healthy",
        "service": "kubernetes-api-simulator",
        "version": "1.0.0"
    }

@app.get("/pods")
def get_pods(namespace: Optional[str] = None) -> Dict[str, Any]:
    """Get pods, optionally filtered by namespace"""
    if namespace:
        filtered_pods = [pod for pod in PODS_DATA["pods"] if pod["namespace"] == namespace]
        return {"pods": filtered_pods}
    return PODS_DATA

@app.get("/pods/{pod_name}")
def get_pod_details(pod_name: str) -> Dict[str, Any]:
    """Get detailed information about a specific pod"""
    for pod in PODS_DATA["pods"]:
        if pod["name"] == pod_name:
            return pod
    raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")

print("✅ FastAPI backend created with realistic Kubernetes data")

In [ ]:
# Start FastAPI server with proper error handling
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
SERVER_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"

def start_server():
    """Start FastAPI server in background thread"""
    try:
        uvicorn.run(
            app, 
            host=SERVER_HOST, 
            port=SERVER_PORT, 
            log_level="error",
            access_log=False
        )
    except Exception as e:
        print(f"❌ Server startup failed: {e}")

def wait_for_server(max_attempts: int = 10, delay: float = 1.0) -> bool:
    """Wait for server to become available"""
    for attempt in range(max_attempts):
        try:
            response = requests.get(f"{SERVER_URL}/health", timeout=2)
            if response.status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(delay)
    return False

# Start server in background thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for server to be ready
print("🚀 Starting FastAPI server...")
if wait_for_server():
    # Verify server functionality
    try:
        health_response = requests.get(f"{SERVER_URL}/health", timeout=5)
        pods_response = requests.get(f"{SERVER_URL}/pods", timeout=5)
        
        if health_response.status_code == 200 and pods_response.status_code == 200:
            health_data = health_response.json()
            pods_data = pods_response.json()
            
            print(f"✅ Backend server running at {SERVER_URL}")
            print(f"   Health status: {health_data['status']}")
            print(f"   Pods available: {len(pods_data['pods'])}")
        else:
            print(f"❌ Server responding but with errors")
            print(f"   Health: {health_response.status_code}")
            print(f"   Pods: {pods_response.status_code}")
    except Exception as e:
        print(f"❌ Server verification failed: {e}")
else:
    print("❌ Server failed to start within timeout period")

## Step 3: Create Strands Agent Tool

Define a tool that the SRE agent can use to investigate Kubernetes pod issues.

In [ ]:
@tool
def get_pod_status(namespace: str = "production") -> str:
    """
    Get comprehensive status information for Kubernetes pods in the specified namespace.
    
    This tool queries the Kubernetes API simulator to retrieve detailed pod information
    including health status, resource usage, container details, and recent events.
    
    Args:
        namespace: Kubernetes namespace to query (default: production)
        
    Returns:
        Formatted string containing comprehensive pod status information including:
        - Pod health and readiness status
        - Resource usage (CPU and memory)
        - Container status and restart information
        - Recent events and error messages
        - Resource limits and requests
    """
    try:
        # Query the Kubernetes API simulator
        response = requests.get(
            f"{SERVER_URL}/pods",
            params={"namespace": namespace} if namespace != "production" else {},
            timeout=10
        )
        response.raise_for_status()
        data = response.json()
        
        # Filter pods by namespace
        filtered_pods = [pod for pod in data["pods"] if pod["namespace"] == namespace]
        
        if not filtered_pods:
            return f"No pods found in namespace '{namespace}'"
        
        # Format comprehensive pod information
        result = f"Found {len(filtered_pods)} pods in '{namespace}' namespace:\n\n"
        
        for pod in filtered_pods:
            # Status indicators
            status_icon = "❌" if not pod["ready"] else "✅"
            
            result += f"{status_icon} Pod: {pod['name']}\n"
            result += f"   Status: {pod['status']} (Ready: {pod['ready']})\n"
            result += f"   Age: {pod.get('age', 'Unknown')}\n"
            result += f"   Node: {pod.get('node', 'Not scheduled')}\n"
            result += f"   Restarts: {pod['restart_count']}\n"
            
            # Resource information
            result += f"   Resource Usage: CPU {pod['cpu_usage']}, Memory {pod['memory_usage']}\n"
            
            if pod.get('resource_limits'):
                limits = pod['resource_limits']
                result += f"   Resource Limits: CPU {limits.get('cpu', 'N/A')}, Memory {limits.get('memory', 'N/A')}\n"
            
            if pod.get('last_restart'):
                result += f"   Last Restart: {pod['last_restart']}\n"
            
            # Container details
            if pod.get('containers'):
                result += f"   Containers:\n"
                for container in pod['containers']:
                    result += f"     - {container['name']}: {container['status']} ({container['reason']})\n"
                    if container.get('exit_code'):
                        result += f"       Exit Code: {container['exit_code']}\n"
                    if container.get('message'):
                        result += f"       Message: {container['message']}\n"
            
            # Recent events (show last 4 events for better context)
            if pod.get('events'):
                result += f"   Recent Events:\n"
                for event in pod['events'][-4:]:
                    result += f"     - {event}\n"
            
            result += "\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {str(e)}\nPlease ensure the backend server is running."
    except Exception as e:
        return f"Unexpected error occurred: {str(e)}"

print("✅ Strands tool function created: get_pod_status()")

## Step 4: Initialize Amazon Bedrock Agent

Create a Strands Agent using Amazon Bedrock's Claude 3 Haiku model with professional SRE capabilities.

In [ ]:
# Initialize Bedrock model with error handling and validation
MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"


try:
    # Create Bedrock model instance
    model = BedrockModel(model_id=MODEL_ID, region=REGION)
    
    # Create Strands Agent with comprehensive SRE system prompt
    agent = Agent(
        model=model,
        tools=[get_pod_status],
        system_prompt="""You are an SRE specializing in Kubernetes. Investigate production issues, identify root causes, and provide clear, 
                        actionable solutions. Focus on resource usage, container restarts, events, and error messages. Respond with impact assessment, 
                        recommended kubectl commands, prioritized fixes, and prevention tips. Be concise and technical."""
                )
    
    print(f"✅ Strands Agent initialized successfully with the {MODEL_ID} model")
    
except Exception as e:
    print(f"❌ Failed to initialize Strands Agent: {e}")
    agent = None
    raise

## Step 5: Execute Production Incident Investigation

Simulate a critical production incident and let the AI agent investigate and provide recommendations.

In [ ]:
if agent:
    print("🚨 PRODUCTION INCIDENT SIMULATION\n" + "="*45)
    print("ALERT: Payment service experiencing issues")

    start_time = time.time()
    incident_description = (
        "URGENT PRODUCTION INCIDENT:\n\n"
        "Payment service is down. Users cannot complete purchases. "
        "Payment API returns 503. High error rates. "
        "Please investigate, find the root cause, resolve, and suggest prevention. "
        "This is a P1 revenue-impacting incident."
    )

    try:
        response = agent(incident_description)
        investigation_time = round(time.time() - start_time, 1)
        print(f"⚡ Investigation completed in {investigation_time} seconds\n" + "="*70)
        print("AI AGENT INVESTIGATION RESULTS\n" + "="*70)
        # Print response simply
        if hasattr(response, 'content'):
            print(response.content)
        elif hasattr(response, 'message'):
            print(response.message)
        else:
            print(str(response))
        print("\n" + "="*70)
        investigation_results = {
            'duration': investigation_time,
            'response': response,
            'success': True
        }
    except Exception as e:
        print(f"❌ Investigation failed: {e}")
        investigation_results = {
            'duration': 0,
            'response': None,
            'success': False,
            'error': str(e)
        }
else:
    print("❌ Cannot run investigation - Strands Agent not initialized")
    investigation_results = {'success': False, 'error': 'Agent not initialized'}

## Step 6: Performance Analysis and Validation

Analyze the SRE agent's performance and validate its understanding of the infrastructure issues.

In [ ]:
if investigation_results.get('success'):
    duration = investigation_results['duration']
    print("Investigation took", duration, "seconds")
    print("✅ Agent worked as expected")
else:
    print("❌ Investigation failed - cannot perform analysis")
    if 'error' in investigation_results:
        print(f"Error: {investigation_results['error']}")

## Step 7: Prepare for Next Steps

Preserve all resources and data for use in the next notebook

In [ ]:
# Store variables for next notebook
workshop_data = {
    'model_id': MODEL_ID,
    'aws_region': REGION,
    'server_url': SERVER_URL,
    'pod_data_schema': PODS_DATA,
    'investigation_results': investigation_results
}

# Save to file for persistence
import json
with open('workshop_00_data.json', 'w') as f:
    json.dump({
        'model_id': MODEL_ID,
        'aws_region': REGION,
        'server_url': SERVER_URL,
        'success': investigation_results.get('success', False)
    }, f, indent=2)

print("✅ Workshop data saved for next notebook")

## Summary and Key Takeaways

### What You Accomplished

In this notebook module, you successfully:

2. **Infrastructure Simulation**: Created a realistic Kubernetes API simulator
3. **AI Agent Development**: Built a Strands Agent with Amazon Bedrock
4. **Tool Integration**: Implemented the @tool decorator pattern
5. **Incident Response**: Demonstrated AI-powered troubleshooting
6. **Performance Analysis**: Measured speed and cost improvements

### Next Steps


---
**Congratulations!** You've completed the Single Tool SRE Agent workshop. You're now ready to build more sophisticated AI-powered infrastructure automation solutions.